# LOGOS prototype 5: does the guardian know what it does not know? (NASA C-MAPSS)

Question: the warning model learns from one kind of engine fault (FD001). When it meets engines with a fault type it never saw (FD003), it stays silent on many of them. Can a separate "do I recognise this?" check tell us, **live and before failure**, which engines the warning model cannot be trusted on?

- The guardian is trained only on FD001 engines 1-80. FD003 contains a fault type FD001 does not have (per NASA's documentation).
- Runtime: Colab, CPU. Each analysis cell takes 1 to 3 minutes.
- Honest limits are at the end.

## Step 1: download the data

In [ ]:
import urllib.request, zipfile, os
import numpy as np, pandas as pd

url = "https://kr0k0tsch.de/rul-datasets/CMAPSSData.zip"
if not os.path.exists("cmapss"):
    urllib.request.urlretrieve(url, "cmapss.zip")
    with zipfile.ZipFile("cmapss.zip") as z:
        z.extractall("cmapss")
print("OK:", sorted(os.listdir("cmapss"))[:5])

## Step 2: what happens outside the guardian's experience (silent misses) and a first familiarity check

In [ ]:
import glob
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

H, W, L = 30, 20, 60
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
sensors = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12", "s13", "s14", "s15", "s17", "s20", "s21"]
d = len(sensors)

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]; m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)]); idx.append(t - 1)
    return np.array(feats), np.array(idx)

def load(name, units=None):
    data = pd.read_csv(glob.glob("cmapss/**/" + name, recursive=True)[0], sep=r"\s+", header=None, names=cols)
    F, y, ep, tp, ft = [], [], [], [], {}
    for u in (units if units is not None else sorted(data["unit"].unique())):
        g = data[data["unit"] == u]; f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); ft[u] = t_f
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    F, y, ep, tp = np.vstack(F), np.concatenate(y), np.concatenate(ep), np.concatenate(tp)
    tf = np.array([ft[u] for u in ep], dtype=float)
    P = dict(ep=ep, t=tp, tf=tf, healthy=~(tp >= tf - L), win=(tp < tf) & (tp >= tf - L), n_fail=len(ft), per1000=(~(tp >= tf - L)).sum() / 1000.0)
    return F, y, P

def evaluate_alarm(score, P, thr, gap=10):
    alarm = score > thr
    m = alarm & P["healthy"]; e_, t_ = P["ep"][m], P["t"][m]
    events = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > gap)).sum()) if len(t_) else 0
    mw = alarm & P["win"]
    warned = len(np.unique(P["ep"][mw])) if mw.any() else 0
    return warned / P["n_fail"], events / P["per1000"]

# --- the guardian knows ONLY engines 1-80 of FD001 (one kind of fault) ---
Ftr, ytr, Ptr = load("train_FD001.txt", list(range(1, 81)))
Fk, yk, Pk = load("train_FD001.txt", list(range(81, 101)))     # known kind, unseen engines
Fu, yu, Pu = load("train_FD003.txt")                            # FD003: includes a fault type FD001 never had

sc = StandardScaler().fit(Ftr)
rng = np.random.default_rng(0)
models = []
for k in range(5):                                              # 5 models trained on resampled engines
    ids = rng.choice(len(Ftr), len(Ftr), replace=True)
    models.append(LogisticRegression(max_iter=3000).fit(sc.transform(Ftr[ids]), ytr[ids]))
risk = lambda F_: np.mean([m.predict_proba(sc.transform(F_))[:, 1] for m in models], axis=0)

lo, hi = 0.0, 1.0
s_tr = risk(Ftr)
for _ in range(30):                                             # alarm level: 1 false alarm per 1000 healthy cycles on training engines
    mid = (lo + hi) / 2
    if evaluate_alarm(s_tr, Ptr, mid)[1] > 1: lo = mid
    else: hi = mid
thr = hi

print("1) Same alarm level, applied to engines the guardian knows vs engines with an unseen fault type")
for nm, F_, P_ in (("known kind (FD001, unseen engines)", Fk, Pk), ("partly unknown (FD003)", Fu, Pu)):
    w_, fa = evaluate_alarm(risk(F_), P_, thr)
    print("  %-38s warned %3.0f%% of engines | false alarms %.1f per 1000 healthy cycles" % (nm, 100 * w_, fa))

# "Do I recognise this?" : how far is a situation from everything seen in training (PCA distance)
pca = PCA(n_components=10).fit(sc.transform(Ftr))
var = pca.explained_variance_
fam = lambda F_: (pca.transform(sc.transform(F_)) ** 2 / var).sum(1)
f_tr, f_k, f_u = fam(Ftr), fam(Fk), fam(Fu)
cut = np.percentile(f_tr, 99)
print("\n2) 'Do I recognise this?' check (distance from what the guardian was trained on)")
print("  share of situations flagged as unfamiliar:  known kind %.0f%%  |  FD003 %.0f%%" % (100 * np.mean(f_k > cut), 100 * np.mean(f_u > cut)))
print("  AUROC for telling FD003 from known engines: %.3f  (0.5 = cannot tell, 1.0 = perfect)" % roc_auc_score(np.r_[np.zeros(len(f_k)), np.ones(len(f_u))], np.r_[f_k, f_u]))
print("\nPlain-language rule for the report: if a situation is flagged as unfamiliar, LOGOS says")
print("  'I do not recognise this situation. My warning score is unreliable here. A human must check.'")

Reading (our run): on engines of the known kind the warning model warned 100%; on FD003 only 56%, with fewer false alarms: it stayed silent rather than noisy. A simple row-level familiarity check separated FD003 from known engines only weakly (AUROC 0.668).

## Step 3: engine by engine, three familiarity checks

In [ ]:
import glob
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

H, W, L = 30, 20, 60
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
sensors = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12", "s13", "s14", "s15", "s17", "s20", "s21"]
d = len(sensors)

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]; m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)]); idx.append(t - 1)
    return np.array(feats), np.array(idx)

def load(name, units=None):
    data = pd.read_csv(glob.glob("cmapss/**/" + name, recursive=True)[0], sep=r"\s+", header=None, names=cols)
    F, y, ep, tp, ft = [], [], [], [], {}
    for u in (units if units is not None else sorted(data["unit"].unique())):
        g = data[data["unit"] == u]; f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); ft[u] = t_f
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    F, y, ep, tp = np.vstack(F), np.concatenate(y), np.concatenate(ep), np.concatenate(tp)
    tf = np.array([ft[u] for u in ep], dtype=float)
    P = dict(ep=ep, t=tp, tf=tf, healthy=~(tp >= tf - L), win=(tp < tf) & (tp >= tf - L), n_fail=len(ft), per1000=(~(tp >= tf - L)).sum() / 1000.0)
    return F, y, P

def evaluate_alarm(score, P, thr, gap=10):
    alarm = score > thr
    m = alarm & P["healthy"]; e_, t_ = P["ep"][m], P["t"][m]
    events = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > gap)).sum()) if len(t_) else 0
    mw = alarm & P["win"]
    warned = len(np.unique(P["ep"][mw])) if mw.any() else 0
    return warned / P["n_fail"], events / P["per1000"]

# --- the guardian knows ONLY engines 1-80 of FD001 (one kind of fault) ---
Ftr, ytr, Ptr = load("train_FD001.txt", list(range(1, 81)))
Fk, yk, Pk = load("train_FD001.txt", list(range(81, 101)))     # known kind, unseen engines
Fu, yu, Pu = load("train_FD003.txt")                            # FD003: includes a fault type FD001 never had

sc = StandardScaler().fit(Ftr)
rng = np.random.default_rng(0)
models = []
for k in range(5):                                              # 5 models trained on resampled engines
    ids = rng.choice(len(Ftr), len(Ftr), replace=True)
    models.append(LogisticRegression(max_iter=3000).fit(sc.transform(Ftr[ids]), ytr[ids]))
risk = lambda F_: np.mean([m.predict_proba(sc.transform(F_))[:, 1] for m in models], axis=0)

lo, hi = 0.0, 1.0
s_tr = risk(Ftr)
for _ in range(30):                                             # alarm level: 1 false alarm per 1000 healthy cycles on training engines
    mid = (lo + hi) / 2
    if evaluate_alarm(s_tr, Ptr, mid)[1] > 1: lo = mid
    else: hi = mid
thr = hi

# three ways to ask "do I recognise this situation?"
from sklearn.neighbors import NearestNeighbors
Ztr = sc.transform(Ftr)
pca = PCA(n_components=10).fit(Ztr); var = pca.explained_variance_
sub = Ztr[np.random.default_rng(1).choice(len(Ztr), min(8000, len(Ztr)), replace=False)]
nn = NearestNeighbors(n_neighbors=10).fit(sub)
fam = {
    "A: PCA distance": lambda F_: (pca.transform(sc.transform(F_)) ** 2 / var).sum(1),
    "B: nearest-neighbour distance": lambda F_: nn.kneighbors(sc.transform(F_))[0].mean(1),
    "C: largest single deviation": lambda F_: np.abs(sc.transform(F_)).max(1),
}
cuts = {k: np.percentile(f(Ftr), 99) for k, f in fam.items()}

# engine by engine: which FD003 engines did the guardian MISS, and did a familiarity check warn about them?
auc = lambda yy, ss: roc_auc_score(yy, ss) if len(set(yy)) > 1 else 0.5
alarm = risk(Fu) > thr
eng = np.unique(Pu["ep"])
missed = np.array([not (alarm & Pu["win"] & (Pu["ep"] == e)).any() for e in eng])
print("FD003 engines: %d | warned %d | MISSED %d" % (len(eng), (~missed).sum(), missed.sum()))
print("\nCan a familiarity check tell us which engines will be missed? (AUROC: 0.5 = no, 1.0 = perfectly)")
shares = {}
for k, f in fam.items():
    flag = f(Fu) > cuts[k]
    shares[k] = np.array([flag[Pu["ep"] == e].mean() for e in eng])
    print("  %-32s AUROC = %.3f" % (k, auc(missed, shares[k])))

best = max(shares, key=lambda k: auc(missed, shares[k]))
sh = shares[best]; cut_e = np.median(sh)
fl = sh > cut_e
print("\nUsing the best check (%s): engines with more unfamiliar situations than the median are 'flagged'" % best)
print("  flagged engines   : %d, of which MISSED by the warning model: %.0f%%" % (fl.sum(), 100 * missed[fl].mean() if fl.any() else 0))
print("  unflagged engines : %d, of which MISSED by the warning model: %.0f%%" % ((~fl).sum(), 100 * missed[~fl].mean() if (~fl).any() else 0))
print("\nIf flagged engines are missed far more often, LOGOS can say: 'do not trust my silence on this engine, a human must check.'")

Reading (our run): counted over the whole life of each engine, all three checks separated the engines the warning model missed from the ones it caught almost perfectly (AUROC about 1.0). Caution: this uses the whole life of the engine, including the moments close to failure, and the cut at the median happened to suit a roughly 50/50 mix.

## Step 4: the live version (only data up to now; rule fixed before seeing FD003)

In [ ]:
import glob
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

H, W, L = 30, 20, 60
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
sensors = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12", "s13", "s14", "s15", "s17", "s20", "s21"]
d = len(sensors)

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]; m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)]); idx.append(t - 1)
    return np.array(feats), np.array(idx)

def load(name, units=None):
    data = pd.read_csv(glob.glob("cmapss/**/" + name, recursive=True)[0], sep=r"\s+", header=None, names=cols)
    F, y, ep, tp, ft = [], [], [], [], {}
    for u in (units if units is not None else sorted(data["unit"].unique())):
        g = data[data["unit"] == u]; f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); ft[u] = t_f
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    F, y, ep, tp = np.vstack(F), np.concatenate(y), np.concatenate(ep), np.concatenate(tp)
    tf = np.array([ft[u] for u in ep], dtype=float)
    P = dict(ep=ep, t=tp, tf=tf, healthy=~(tp >= tf - L), win=(tp < tf) & (tp >= tf - L), n_fail=len(ft), per1000=(~(tp >= tf - L)).sum() / 1000.0)
    return F, y, P

def evaluate_alarm(score, P, thr, gap=10):
    alarm = score > thr
    m = alarm & P["healthy"]; e_, t_ = P["ep"][m], P["t"][m]
    events = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > gap)).sum()) if len(t_) else 0
    mw = alarm & P["win"]
    warned = len(np.unique(P["ep"][mw])) if mw.any() else 0
    return warned / P["n_fail"], events / P["per1000"]

# --- the guardian knows ONLY engines 1-80 of FD001 (one kind of fault) ---
Ftr, ytr, Ptr = load("train_FD001.txt", list(range(1, 81)))
Fk, yk, Pk = load("train_FD001.txt", list(range(81, 101)))     # known kind, unseen engines
Fu, yu, Pu = load("train_FD003.txt")                            # FD003: includes a fault type FD001 never had

sc = StandardScaler().fit(Ftr)
rng = np.random.default_rng(0)
models = []
for k in range(5):                                              # 5 models trained on resampled engines
    ids = rng.choice(len(Ftr), len(Ftr), replace=True)
    models.append(LogisticRegression(max_iter=3000).fit(sc.transform(Ftr[ids]), ytr[ids]))
risk = lambda F_: np.mean([m.predict_proba(sc.transform(F_))[:, 1] for m in models], axis=0)

lo, hi = 0.0, 1.0
s_tr = risk(Ftr)
for _ in range(30):                                             # alarm level: 1 false alarm per 1000 healthy cycles on training engines
    mid = (lo + hi) / 2
    if evaluate_alarm(s_tr, Ptr, mid)[1] > 1: lo = mid
    else: hi = mid
thr = hi

# three ways to ask "do I recognise this situation?"
from sklearn.neighbors import NearestNeighbors
Ztr = sc.transform(Ftr)
pca = PCA(n_components=10).fit(Ztr); var = pca.explained_variance_
sub = Ztr[np.random.default_rng(1).choice(len(Ztr), min(8000, len(Ztr)), replace=False)]
nn = NearestNeighbors(n_neighbors=10).fit(sub)
fam = {
    "A: PCA distance": lambda F_: (pca.transform(sc.transform(F_)) ** 2 / var).sum(1),
    "B: nearest-neighbour distance": lambda F_: nn.kneighbors(sc.transform(F_))[0].mean(1),
    "C: largest single deviation": lambda F_: np.abs(sc.transform(F_)).max(1),
}
cuts = {k: np.percentile(f(Ftr), 99) for k, f in fam.items()}

# ---- LIVE version: only data up to the current cycle is used, and the rule is fixed BEFORE seeing FD003 ----
K = 30                                                           # look at the last 30 cycles
fB = fam["B: nearest-neighbour distance"]
ind_k, ind_u = fB(Fk) > cuts["B: nearest-neighbour distance"], fB(Fu) > cuts["B: nearest-neighbour distance"]

def first_flag(ind, P, e, s):
    m = P["ep"] == e
    i, t = ind[m].astype(float), P["t"][m]
    roll = np.convolve(i, np.ones(K) / K, mode="full")[:len(i)]
    ok = np.where((roll >= s) & (np.arange(len(i)) >= K - 1))[0]
    return t[ok[0]] if len(ok) else None

# choose the rule on known engines FD001 81-100 (never used for training): at most 5% of them may ever be flagged
eng_k = np.unique(Pk["ep"])
rule = 1.0
for s_ in np.arange(0.2, 1.01, 0.05):
    if np.mean([first_flag(ind_k, Pk, e, s_) is not None for e in eng_k]) <= 0.05:
        rule = s_; break
print("rule chosen on known engines: flag an engine when >= %.0f%% of its last %d cycles look unfamiliar" % (100 * rule, K))
print("known engines ever flagged: %.0f%% (this is the false-flag rate)" % (100 * np.mean([first_flag(ind_k, Pk, e, rule) is not None for e in eng_k])))

eng = np.unique(Pu["ep"])
alarm = risk(Fu) > thr
missed = np.array([not (alarm & Pu["win"] & (Pu["ep"] == e)).any() for e in eng])
tfail = np.array([Pu["tf"][Pu["ep"] == e][0] for e in eng])
tflag = [first_flag(ind_u, Pu, e, rule) for e in eng]
lead = np.array([(tf_ - t_) if t_ is not None else np.nan for tf_, t_ in zip(tfail, tflag)])
flagged = ~np.isnan(lead)

print("\nFD003: %d engines | warned by the model %d | MISSED %d" % (len(eng), (~missed).sum(), missed.sum()))
print("Engines the model MISSED:")
print("  flagged 'unfamiliar' before the failure : %.0f%%" % (100 * flagged[missed].mean() if missed.any() else 0))
print("  flagged at least 60 cycles before failure: %.0f%%" % (100 * np.mean(lead[missed] >= 60) if missed.any() else 0))
if missed.any() and flagged[missed].any():
    print("  median time of the flag: %.0f cycles before failure" % np.nanmedian(lead[missed]))
print("Engines the model WARNED about, flagged anyway: %.0f%%" % (100 * flagged[~missed].mean() if (~missed).any() else 0))
print("\nShare of MISSED engines already flagged by cycle:")
for c in (60, 90, 120, 150):
    got = [(t_ is not None and t_ <= c) for t_ in np.array(tflag, dtype=object)[missed]]
    print("  cycle %3d: %.0f%%" % (c, 100 * np.mean(got) if len(got) else 0))
print("\nIf most missed engines are flagged well before they fail, LOGOS can say it early:")
print("  'I do not recognise how this engine is behaving. My silence is not a good sign. A human must check.'")

Reading (our run): the rule was chosen on 20 known engines never used for training (5% flagged). On FD003 every one of the 44 engines the warning model missed was flagged as unfamiliar before failure, 98% of them at least 60 cycles early (median 182 cycles). 11% of the engines the model did warn about were flagged as well.

Caution found afterwards: in Steps 3 and 4 the distance cut was computed on the training rows, which are also the reference points of the nearest-neighbour check, so the cut was biased. Step 5 fixes this and repeats the test.

## Step 5: repeated, corrected test on four dataset pairs (3 random splits each)

In [ ]:
import glob, time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

H, W, L, K = 30, 20, 60, 30
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
sensors = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12", "s13", "s14", "s15", "s17", "s20", "s21"]

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]; m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)]); idx.append(t - 1)
    return np.array(feats), np.array(idx)

D = {}                                                    # all four files, features computed once
t0 = time.time()
for name in ("FD001", "FD002", "FD003", "FD004"):
    data = pd.read_csv(glob.glob("cmapss/**/train_%s.txt" % name, recursive=True)[0], sep=r"\s+", header=None, names=cols)
    F, y, ep, tp, tf = [], [], [], [], []
    for u, g in data.groupby("unit"):
        f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); tf.append(np.full(len(idx), float(t_f)))
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    D[name] = dict(F=np.vstack(F), y=np.concatenate(y), ep=np.concatenate(ep), t=np.concatenate(tp), tf=np.concatenate(tf))
    print(name, "ready (%d engines)" % len(np.unique(D[name]["ep"])), "| %.0fs" % (time.time() - t0))

def part(name, units):
    d_ = D[name]; m = np.isin(d_["ep"], units)
    P = dict(ep=d_["ep"][m], t=d_["t"][m], tf=d_["tf"][m])
    P["healthy"] = ~(P["t"] >= P["tf"] - L); P["win"] = (P["t"] < P["tf"]) & (P["t"] >= P["tf"] - L)
    P["n_fail"] = len(units); P["per1000"] = P["healthy"].sum() / 1000.0
    return d_["F"][m], d_["y"][m], P

def evaluate_alarm(score, P, thr, gap=10):
    alarm = score > thr
    m = alarm & P["healthy"]; e_, t_ = P["ep"][m], P["t"][m]
    events = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > gap)).sum()) if len(t_) else 0
    return events / P["per1000"]

def first_flag(ind, P, e, s):
    m = P["ep"] == e
    i, t = ind[m].astype(float), P["t"][m]
    roll = np.convolve(i, np.ones(K) / K, mode="full")[:len(i)]
    ok = np.where((roll >= s) & (np.arange(len(i)) >= K - 1))[0]
    return t[ok[0]] if len(ok) else None

def run_pair(known, new, seed):
    rng = np.random.default_rng(seed)
    units = np.array(sorted(np.unique(D[known]["ep"]))); rng.shuffle(units)
    a, b = int(0.6 * len(units)), int(0.8 * len(units))
    tr_u, ca_u, cb_u = units[:a], units[a:b], units[b:]          # train | set A (distance cut) | set B (flag rule)
    Ftr, ytr, Ptr = part(known, tr_u); Fca, yca, Pca = part(known, ca_u); Fcb, ycb, Pcb = part(known, cb_u)
    Fab, yab, Pab = part(known, np.r_[ca_u, cb_u])
    Fn, yn, Pn = part(new, np.unique(D[new]["ep"]))
    sc = StandardScaler().fit(Ftr); Ztr = sc.transform(Ftr)
    models = []
    for k in range(5):
        ids = rng.choice(len(Ftr), len(Ftr), replace=True)
        models.append(LogisticRegression(max_iter=3000).fit(Ztr[ids], ytr[ids]))
    risk = lambda F_: np.mean([m.predict_proba(sc.transform(F_))[:, 1] for m in models], axis=0)
    lo, hi = 0.0, 1.0; s_ab = risk(Fab)                     # warning level set on engines never used for training
    for _ in range(30):
        mid = (lo + hi) / 2
        if evaluate_alarm(s_ab, Pab, mid) > 1: lo = mid
        else: hi = mid
    thr = hi
    sub = Ztr[rng.choice(len(Ztr), min(8000, len(Ztr)), replace=False)]
    nn = NearestNeighbors(n_neighbors=10).fit(sub)
    dist = lambda F_: nn.kneighbors(sc.transform(F_))[0].mean(1)
    cut = np.percentile(dist(Fca), 99)                      # distance cut from UNSEEN known engines (not training rows)
    ind_cb, ind_n = dist(Fcb) > cut, dist(Fn) > cut
    ecb = np.unique(Pcb["ep"]); rule = 1.0
    for s_ in np.arange(0.2, 1.01, 0.05):
        if np.mean([first_flag(ind_cb, Pcb, e, s_) is not None for e in ecb]) <= 0.05:
            rule = s_; break
    eng = np.unique(Pn["ep"]); alarm = risk(Fn) > thr
    missed = np.array([not (alarm & Pn["win"] & (Pn["ep"] == e)).any() for e in eng])
    tfail = np.array([Pn["tf"][Pn["ep"] == e][0] for e in eng])
    tflag = [first_flag(ind_n, Pn, e, rule) for e in eng]
    lead = np.array([(x - y_) if y_ is not None else np.nan for x, y_ in zip(tfail, tflag)])
    fl = ~np.isnan(lead)
    return dict(n=len(eng), missed=int(missed.sum()), rule=rule,
                m_flag=100 * fl[missed].mean() if missed.any() else np.nan,
                m_early=100 * np.mean(lead[missed] >= 60) if missed.any() else np.nan,
                w_flag=100 * fl[~missed].mean() if (~missed).any() else np.nan,
                known_flag=100 * np.mean([first_flag(ind_cb, Pcb, e, rule) is not None for e in ecb]))

def show(v): return "n/a" if v != v else "%.0f%%" % v
print("Fixed version: the distance cut now comes from known engines that were NOT used for training.")
print("%-14s %-10s %-7s %-22s %-14s %s" % ("learns -> meets", "missed", "rule", "missed ones flagged", ">=60 cyc early", "warned-but-flagged | known flagged"))
for kn, nw in [("FD001", "FD003"), ("FD003", "FD001"), ("FD002", "FD004"), ("FD004", "FD002")]:
    R = [run_pair(kn, nw, s) for s in range(3)]
    m = lambda k: np.nan if np.all(np.isnan([r[k] for r in R])) else np.nanmean([r[k] for r in R])
    lo_, hi_ = (np.nanmin([r["m_flag"] for r in R]), np.nanmax([r["m_flag"] for r in R])) if R[0]["missed"] else (np.nan, np.nan)
    print("%-14s %-10s %-7s %-22s %-14s %s | %s" % (kn + " -> " + nw, "%.0f of %d" % (np.mean([r["missed"] for r in R]), R[0]["n"]),
          "%.0f%%" % (100 * np.mean([r["rule"] for r in R])), "%s (%s to %s)" % (show(m("m_flag")), show(lo_), show(hi_)),
          show(m("m_early")), show(m("w_flag")), show(m("known_flag"))))

Reading (our run, corrected version):

| learns -> meets | engines missed | missed ones flagged | flagged >=60 cycles early | warned engines flagged | known engines flagged |
|---|---|---|---|---|---|
| FD001 -> FD003 | 44 of 100 | 100% (100 to 100) | 99% | 3% | 5% |
| FD003 -> FD001 | 0 of 100 | n/a | n/a | 3% | 5% |
| FD002 -> FD004 | 101 of 249 | 8% (5 to 12) | 4% | 2% | 3% |
| FD004 -> FD002 | 0 of 260 | n/a | n/a | 4% | 3% |

The familiarity check works on the single-condition pair (FD001 -> FD003) and fails on the six-condition pair (FD002 -> FD004), where the warning model misses 40% of engines and the check catches only 8% of them. Hypothesis: raw sensor values there are dominated by changing operating conditions, so a distance check cannot see the unseen fault. Step 6 tests this by normalising sensors per operating condition.

## Step 6: six-condition pairs again, with sensors normalised per operating condition

In [ ]:
from sklearn.cluster import KMeans

def read(name):
    return pd.read_csv(glob.glob("cmapss/**/train_%s.txt" % name, recursive=True)[0], sep=r"\s+", header=None, names=cols)

raw = {n: read(n) for n in ("FD002", "FD004")}
sets = ["set1", "set2", "set3"]
setsc = StandardScaler().fit(raw["FD002"][sets])
km = KMeans(n_clusters=6, n_init=10, random_state=0).fit(setsc.transform(raw["FD002"][sets]))   # the 6 operating conditions
for n in raw:
    raw[n]["cond"] = km.predict(setsc.transform(raw[n][sets]))
print("operating-condition clusters found:", {n: sorted(np.bincount(raw[n]["cond"], minlength=6).tolist()) for n in raw})

def build_normalised(name, ref_name):
    """Sensors are standardised separately inside each operating condition, using the EARLY life (first 30 cycles)
    of the known dataset only as the 'healthy reference'."""
    ref = raw[ref_name][raw[ref_name]["cycle"] <= 30]
    mu = ref.groupby("cond")[sensors].mean(); sd = ref.groupby("cond")[sensors].std() + 1e-9
    data = raw[name].copy()
    data[sensors] = (data[sensors].values - mu.loc[data["cond"]].values) / sd.loc[data["cond"]].values
    F, y, ep, tp, tf = [], [], [], [], []
    for u, g in data.groupby("unit"):
        f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); tf.append(np.full(len(idx), float(t_f)))
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    return dict(F=np.vstack(F), y=np.concatenate(y), ep=np.concatenate(ep), t=np.concatenate(tp), tf=np.concatenate(tf))

D_saved = D
print("\nSix-condition pairs, sensors normalised per operating condition. 3 splits each.")
print("%-14s %-10s %-7s %-22s %-14s %s" % ("learns -> meets", "missed", "rule", "missed ones flagged", ">=60 cyc early", "warned-but-flagged | known flagged"))
for kn, nw in [("FD002", "FD004"), ("FD004", "FD002")]:
    D = {kn: build_normalised(kn, kn), nw: build_normalised(nw, kn)}
    R = [run_pair(kn, nw, s) for s in range(3)]
    m = lambda k: np.nan if np.all(np.isnan([r[k] for r in R])) else np.nanmean([r[k] for r in R])
    lo_, hi_ = (np.nanmin([r["m_flag"] for r in R]), np.nanmax([r["m_flag"] for r in R])) if R[0]["missed"] else (np.nan, np.nan)
    print("%-14s %-10s %-7s %-22s %-14s %s | %s" % (kn + " -> " + nw, "%.0f of %d" % (np.mean([r["missed"] for r in R]), R[0]["n"]),
          "%.0f%%" % (100 * np.mean([r["rule"] for r in R])), "%s (%s to %s)" % (show(m("m_flag")), show(lo_), show(hi_)),
          show(m("m_early")), show(m("w_flag")), show(m("known_flag"))))
D = D_saved
print("\nCompare with the un-normalised run above (FD002 -> FD004: 101 of 249 missed, only 8% of them flagged).")

Reading (our run):

| learns -> meets | engines missed | missed ones flagged | flagged >=60 cycles early | warned engines flagged | known engines flagged |
|---|---|---|---|---|---|
| FD002 -> FD004 | 92 of 249 | 100% (100 to 100) | 94% | 10% | 4% |
| FD004 -> FD002 | 0 of 260 | n/a | n/a | 5% | 3% |

Normalising per operating condition repaired the failure: the check now flagged every engine the warning model missed (8% before). The hypothesis that operating conditions were masking the unseen fault is supported. Caution: the six condition clusters found by k-means are not equally sized (one is larger, 13458 rows against about 8000), so they may not match the true six conditions exactly.

## Step 7: what a human is told with and without the check (5 splits per pair), and when the first flag comes

In [ ]:
import numpy as np, pandas as pd
import glob, time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

H, W, L, K = 30, 20, 60, 30
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
sensors = ["s2", "s3", "s4", "s7", "s8", "s9", "s11", "s12", "s13", "s14", "s15", "s17", "s20", "s21"]

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]; m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)]); idx.append(t - 1)
    return np.array(feats), np.array(idx)

D = {}                                                    # all four files, features computed once
t0 = time.time()
for name in ("FD001", "FD002", "FD003", "FD004"):
    data = pd.read_csv(glob.glob("cmapss/**/train_%s.txt" % name, recursive=True)[0], sep=r"\s+", header=None, names=cols)
    F, y, ep, tp, tf = [], [], [], [], []
    for u, g in data.groupby("unit"):
        f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); tf.append(np.full(len(idx), float(t_f)))
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    D[name] = dict(F=np.vstack(F), y=np.concatenate(y), ep=np.concatenate(ep), t=np.concatenate(tp), tf=np.concatenate(tf))
    print(name, "ready (%d engines)" % len(np.unique(D[name]["ep"])), "| %.0fs" % (time.time() - t0))

def part(name, units):
    d_ = D[name]; m = np.isin(d_["ep"], units)
    P = dict(ep=d_["ep"][m], t=d_["t"][m], tf=d_["tf"][m])
    P["healthy"] = ~(P["t"] >= P["tf"] - L); P["win"] = (P["t"] < P["tf"]) & (P["t"] >= P["tf"] - L)
    P["n_fail"] = len(units); P["per1000"] = P["healthy"].sum() / 1000.0
    return d_["F"][m], d_["y"][m], P

def evaluate_alarm(score, P, thr, gap=10):
    alarm = score > thr
    m = alarm & P["healthy"]; e_, t_ = P["ep"][m], P["t"][m]
    events = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > gap)).sum()) if len(t_) else 0
    return events / P["per1000"]

def first_flag(ind, P, e, s):
    m = P["ep"] == e
    i, t = ind[m].astype(float), P["t"][m]
    roll = np.convolve(i, np.ones(K) / K, mode="full")[:len(i)]
    ok = np.where((roll >= s) & (np.arange(len(i)) >= K - 1))[0]
    return t[ok[0]] if len(ok) else None


from sklearn.cluster import KMeans

def read(name):
    return pd.read_csv(glob.glob("cmapss/**/train_%s.txt" % name, recursive=True)[0], sep=r"\s+", header=None, names=cols)

raw = {n: read(n) for n in ("FD002", "FD004")}
sets = ["set1", "set2", "set3"]
setsc = StandardScaler().fit(raw["FD002"][sets])
km = KMeans(n_clusters=6, n_init=10, random_state=0).fit(setsc.transform(raw["FD002"][sets]))   # the 6 operating conditions
for n in raw:
    raw[n]["cond"] = km.predict(setsc.transform(raw[n][sets]))
print("operating-condition clusters found:", {n: sorted(np.bincount(raw[n]["cond"], minlength=6).tolist()) for n in raw})

def build_normalised(name, ref_name):
    """Sensors are standardised separately inside each operating condition, using the EARLY life (first 30 cycles)
    of the known dataset only as the 'healthy reference'."""
    ref = raw[ref_name][raw[ref_name]["cycle"] <= 30]
    mu = ref.groupby("cond")[sensors].mean(); sd = ref.groupby("cond")[sensors].std() + 1e-9
    data = raw[name].copy()
    data[sensors] = (data[sensors].values - mu.loc[data["cond"]].values) / sd.loc[data["cond"]].values
    F, y, ep, tp, tf = [], [], [], [], []
    for u, g in data.groupby("unit"):
        f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); tf.append(np.full(len(idx), float(t_f)))
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    return dict(F=np.vstack(F), y=np.concatenate(y), ep=np.concatenate(ep), t=np.concatenate(tp), tf=np.concatenate(tf))


def run_pair2(known, new, seed):
    rng = np.random.default_rng(seed)
    units = np.array(sorted(np.unique(D[known]["ep"]))); rng.shuffle(units)
    a, b = int(0.6 * len(units)), int(0.8 * len(units))
    tr_u, ca_u, cb_u = units[:a], units[a:b], units[b:]          # train | set A (distance cut) | set B (flag rule)
    Ftr, ytr, Ptr = part(known, tr_u); Fca, yca, Pca = part(known, ca_u); Fcb, ycb, Pcb = part(known, cb_u)
    Fab, yab, Pab = part(known, np.r_[ca_u, cb_u])
    Fn, yn, Pn = part(new, np.unique(D[new]["ep"]))
    sc = StandardScaler().fit(Ftr); Ztr = sc.transform(Ftr)
    models = []
    for k in range(5):
        ids = rng.choice(len(Ftr), len(Ftr), replace=True)
        models.append(LogisticRegression(max_iter=3000).fit(Ztr[ids], ytr[ids]))
    risk = lambda F_: np.mean([m.predict_proba(sc.transform(F_))[:, 1] for m in models], axis=0)
    lo, hi = 0.0, 1.0; s_ab = risk(Fab)                     # warning level set on engines never used for training
    for _ in range(30):
        mid = (lo + hi) / 2
        if evaluate_alarm(s_ab, Pab, mid) > 1: lo = mid
        else: hi = mid
    thr = hi
    sub = Ztr[rng.choice(len(Ztr), min(8000, len(Ztr)), replace=False)]
    nn = NearestNeighbors(n_neighbors=10).fit(sub)
    dist = lambda F_: nn.kneighbors(sc.transform(F_))[0].mean(1)
    cut = np.percentile(dist(Fca), 99)                      # distance cut from UNSEEN known engines (not training rows)
    ind_cb, ind_n = dist(Fcb) > cut, dist(Fn) > cut
    ecb = np.unique(Pcb["ep"]); rule = 1.0
    for s_ in np.arange(0.2, 1.01, 0.05):
        if np.mean([first_flag(ind_cb, Pcb, e, s_) is not None for e in ecb]) <= 0.05:
            rule = s_; break
    eng = np.unique(Pn["ep"]); alarm = risk(Fn) > thr
    missed = np.array([not (alarm & Pn["win"] & (Pn["ep"] == e)).any() for e in eng])
    tfail = np.array([Pn["tf"][Pn["ep"] == e][0] for e in eng])
    tflag = [first_flag(ind_n, Pn, e, rule) for e in eng]
    lead = np.array([(x - y_) if y_ is not None else np.nan for x, y_ in zip(tfail, tflag)])
    fl = ~np.isnan(lead)
    tfl = np.array([np.nan if t_ is None else float(t_) for t_ in tflag])
    mf = missed & fl
    return dict(n=len(eng), missed=int(missed.sum()), rule=rule,
                cover_alarm=100 * np.mean(~missed), cover_both=100 * np.mean((~missed) | fl),
                first_cycle=np.nanmedian(tfl[mf]) if mf.any() else np.nan,
                first_life=100 * np.nanmedian(tfl[mf] / tfail[mf]) if mf.any() else np.nan,
                by_cycle_60=100 * np.mean(tfl[missed] <= 60) if missed.any() else np.nan,
                m_flag=100 * fl[missed].mean() if missed.any() else np.nan,
                m_early=100 * np.mean(lead[missed] >= 60) if missed.any() else np.nan,
                w_flag=100 * fl[~missed].mean() if (~missed).any() else np.nan,
                known_flag=100 * np.mean([first_flag(ind_cb, Pcb, e, rule) is not None for e in ecb]))


D_orig = D
def one(v): return "n/a" if v != v else "%.0f" % v
print("What a human is told, with and without the 'do I recognise this?' check. 5 splits per pair.")
print("%-14s %-24s %-22s %-24s %s" % ("learns -> meets", "engines with a warning", "...with warning OR flag", "first flag (median cycle,", "extra flags on"))
print("%-14s %-24s %-22s %-24s %s" % ("", "(alarm alone)", "(system with check)", "% of the engine's life)", "OK engines"))
for kn, nw, norm in [("FD001", "FD003", False), ("FD003", "FD001", False), ("FD002", "FD004", True), ("FD004", "FD002", True)]:
    D = {kn: build_normalised(kn, kn), nw: build_normalised(nw, kn)} if norm else D_orig
    R = [run_pair2(kn, nw, s) for s in range(5)]
    g = lambda k: np.nan if np.all(np.isnan([r[k] for r in R])) else np.nanmean([r[k] for r in R])
    print("%-14s %-24s %-22s %-24s %s" % (kn + " -> " + nw, "%s%% (of %d)" % (one(g("cover_alarm")), R[0]["n"]),
          "%s%%" % one(g("cover_both")), "cycle %s, %s%% of life" % (one(g("first_cycle")), one(g("first_life"))),
          "%s%% of warned | %s%% of known" % (one(g("w_flag")), one(g("known_flag")))))
D = D_orig
print("\nIf the first flag comes at about cycle 50 for every pair, the check may mostly notice that an engine looks")
print("different from the start, rather than the moment a new fault begins. Both are useful to a human, but they are different claims.")

Reading (our run):

| learns -> meets | engines with a warning (alarm alone) | with warning OR flag | first flag (median) | extra flags on warned engines | flags on known OK engines |
|---|---|---|---|---|---|
| FD001 -> FD003 | 61% (of 100) | 100% | cycle 48 (25% of life) | 11% | 5% |
| FD003 -> FD001 | 100% (of 100) | 100% | n/a | 2% | 5% |
| FD002 -> FD004 | 62% (of 249) | 100% | cycle 68 (29% of life) | 8% | 3% |
| FD004 -> FD002 | 100% (of 260) | 100% | n/a | 5% | 3% |

Important: with the settings used (20-cycle window, 30-cycle flag window) the earliest possible flag is at about cycle 48. The median first flag is at cycle 48 and 68, i.e. almost immediately. So the check mainly notices that an engine is **of an unfamiliar kind from early in its life**, not the moment a new fault starts. Only about half of the engines in the new dataset were flagged (all the missed ones plus about 10% of the others), so it is not simply "the whole dataset looks different", but this is not proof either. A flag says "I do not recognise this engine, a human should look", not "failure is imminent".

## Limits (please read)

- NASA C-MAPSS is a simulation of jet engines, not cars and not real fleet logs.
- Four pairs were tested (3 splits each). On the two pairs with unseen fault types the check flagged every missed engine (FD001 to FD003; FD002 to FD004 after per-condition normalisation). The other two pairs had no missed engines to find. Two real tests with real misses is a small number.
- The flag appears very early (median 182 cycles before failure). That suggests the check may be noticing that these engines look different from the start, not necessarily the unseen fault itself. NASA does not label the fault type per engine, so we could not verify this.
- The rule is calibrated on a small set of known engines (20 to 50), so the false-flag rate is uncertain.
- Detecting unfamiliar inputs and predicting failure from sensors are both established research areas. The point here is the combination: a system that says "I do not recognise this, my silence is not reassuring, a human must check".
- No safety assessment of any kind.